# Benchmark de Cascata Adaptativa — Marco 1

Validação no Google Colab (GPU NVIDIA T4) dos Componentes B, C, F e G:

- **B** — servidor vLLM com prefix caching e `--logprobs-mode raw_logprobs` (Story 1 do `engine-router-spec`)
- **C** — System 1: decisão tipada de 1 token via `/v1/systemone` (Story 2 do `engine-router-spec`)
- **F** — System 2: CoT + JSON via `/v1/chat/completions` (Story 1 do `system2-evaluator-spec`)
- **G** — benchmark completo: 3 cenários, sweep de α e artefatos em `results/` (Stories 2–3 do `system2-evaluator-spec`)

**Como usar**: em *Runtime → Change runtime type*, selecione **T4 GPU** e rode as células em ordem.

Nota de reprodutibilidade: a documentação do vLLM avisa que resultados podem variar entre requests idênticas (batching/prefix cache). `VLLM_BATCH_INVARIANT=1` **não** é configurado porque o benchmark mede latência realista.


## Setup — clone e dependências

Clona o repo público e instala `requirements-colab.txt` (inclui o vLLM; a instalação demora alguns minutos).

In [ ]:
import os
import subprocess

REPO = "adaptive-cascade-routing"
BRANCH = "004-component-g-evaluator"  # PR #5; trocar para "main" após o merge
REPO_URL = "https://github.com/Grazinascito/adaptive-cascade-routing.git"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO, check=True)

In [ ]:
%cd adaptive-cascade-routing
!git log --oneline -1
!pip install -q -r requirements-colab.txt

In [ ]:
import os
import sys

# Guard contra execução fora de ordem (ex.: runtime reiniciado sem
# rodar o Setup): garante que o cwd é o repo clonado antes de
# registrar src/ no sys.path.
if os.path.exists("adaptive-cascade-routing"):
    os.chdir("adaptive-cascade-routing")
sys.path.insert(0, os.getcwd())

from src.data.loader import format_system1_prompt, get_frozen_eval_set
from src.engine import server, system1, system2
from src.metrics.evaluator import (
    JSON_GATE_TARGET,
    P95_SYSTEM1_TARGET_MS,
    BenchmarkHarness,
    plot_tradeoff_curve,
    save_raw_results,
    sweep_alpha,
)
from src.metrics.summary import compute_summary_metrics, evaluate_kpis

## Componente B — servidor vLLM (Story 1)

Sobe o servidor com `Qwen/Qwen3-4B-AWQ` (AWQ 4-bit, arquitetura `Qwen3ForCausalLM` — requisito da read strategy do `/v1/systemone`), prefix caching e 90% da VRAM (NFR-002: ≤ 14.4 GB da T4 de 16 GB). Na primeira execução o vLLM baixa ~2.7 GB de pesos, por isso o timeout de startup aqui é 1800 s. Se algo falhar, o log completo do servidor fica em `vllm_server.log`.


In [ ]:
started = server.start_server(system1.DEFAULT_MODEL, startup_timeout_s=1800.0)
print("start_server:", started)
print("health_check:", server.health_check())
print("log do servidor:", server.VLLM_LOG_PATH)

In [ ]:
warmup = format_system1_prompt("Um cachorro corre no parque.", "O cachorro está correndo.")
latencies = []
for i in range(10):
    output = system1.evaluate_typed_decision(warmup["user_state"], warmup["choices"])
    latencies.append(output["latency_ms"])
    print(f"warmup {i + 1:>2}: {output['latency_ms']:.1f} ms")

rest_avg = sum(latencies[1:]) / len(latencies[1:])
drop = 100.0 * (latencies[0] - rest_avg) / latencies[0]
print(f"\nqueda média das queries 2-10 vs a 1ª: {drop:.1f}% (Independent Test: >= 70%)")

In [ ]:
!nvidia-smi
print("limite NFR-002: 14.4 GB (90% da T4 de 16 GB) para pesos + KV cache")

## Componente C — System 1 (Story 2)

50 amostras balanceadas do frozen set pelo caminho fast-path. Alvo NFR-001: p95 < 60 ms (valor medido é informativo — depende do hardware do momento); a acurácia é reportada para contexto.

In [ ]:
import statistics

samples = get_frozen_eval_set(sample_limit=50)
print(f"{len(samples)} amostras balanceadas do frozen set")

latencies = []
correct = 0
first_output = None
for sample in samples:
    payload = format_system1_prompt(sample["premise"], sample["hypothesis"])
    output = system1.evaluate_typed_decision(payload["user_state"], payload["choices"])
    latencies.append(output["latency_ms"])
    correct += output["top_choice"] == sample["label"]
    if first_output is None:
        first_output = output

qs = statistics.quantiles(latencies, n=100)
print(f"p50={qs[49]:.1f} ms | p95={qs[94]:.1f} ms | p99={qs[98]:.1f} ms (alvo p95 < 60 ms)")
print(f"acurácia System 1: {correct}/{len(samples)} = {correct / len(samples):.1%}")
print(
    "exemplo de saída:",
    {key: first_output[key] for key in ("top_choice", "top_probability", "entropy")},
)

## Componente F — System 2 (Story 1)

20 pares pelo slow-path (CoT + JSON estrito). Independent Test: ≥ 95% de respostas com JSON válido sem precisar do regex de fallback.

In [ ]:
pairs = get_frozen_eval_set(sample_limit=20)
outputs = [system2.execute_reasoning(s["premise"], s["hypothesis"]) for s in pairs]

valid = sum(not o["fallback_used"] for o in outputs)
print(f"JSON válido sem regex: {valid}/{len(outputs)} = {valid / len(outputs):.1%} (limiar >= 95%)")
lat = [o["latency_ms"] for o in outputs]
print(f"latência System 2 — min {min(lat):.0f} ms / max {max(lat):.0f} ms")
for o in outputs[:2]:
    print("-", o["label"], "| fallback:", o["fallback_used"], "|", o["reasoning"][:120])

## Componente G — benchmark completo (Stories 2–3)

O harness roda os **3 cenários** nas mesmas amostras congeladas (FR-003): System 1 puro (`/v1/systemone`), System 2 puro (`/v1/chat/completions`) e Cascata (α = 0.85, decisão via `route()`). O **sweep de α ∈ {0.50, 0.70, 0.85, 0.90, 0.95}** é recomputado das medições gravadas (replay, sem novas chamadas HTTP; `temperature=0.0` torna a saída do System 2 determinística na mesma sessão). Artefatos em `results/` (`raw_benchmark_data.json`, `latency_cost_tradeoff.png`).



In [ ]:
# Story 2 (Independent Test): dry-run com 10 amostras nos 3 cenários e
# verificação de completude — todo query_id precisa aparecer nos 3.
from pathlib import Path

dry_samples = get_frozen_eval_set(sample_limit=10)
dry_harness = BenchmarkHarness(dry_samples)
try:
    dry_run = dry_harness.run_full_benchmark(confidence_threshold=0.85)
finally:
    dry_harness.close()

ids = {s["id"] for s in dry_samples}
missing = {
    scenario: ids - {r["query_id"] for r in records}
    for scenario, records in dry_run["records"].items()
}
save_raw_results(
    Path("results/raw_benchmark_data_dryrun.json"),
    meta={"model": system1.DEFAULT_MODEL, "frozen_samples": len(dry_samples), "dry_run": True},
    scenarios=dry_run["records"],
    alpha_sweep={},
    summaries={},
    kpis=None,
)

print("== G1: dry-run (10 amostras) ==")
for scenario, records in dry_run["records"].items():
    print(f"  {scenario}: {len(records)} registros, ausentes={sorted(missing[scenario])}")
assert all(not miss for miss in missing.values()), missing
print("OK — todo query_id presente nos 3 cenários; log em results/raw_benchmark_data_dryrun.json")

In [ ]:
# Benchmark completo: 200 amostras congeladas, α padrão do config (0.85).
# HF_DATASETS_OFFLINE=1: o dataset já está em cache das células
# anteriores; a carga cai para milissegundos.
import os

os.environ["HF_DATASETS_OFFLINE"] = "1"

full_samples = get_frozen_eval_set()  # 200 amostras, seed 42 (config congelado)
full_harness = BenchmarkHarness(full_samples)
try:
    full_run = full_harness.run_full_benchmark(confidence_threshold=0.85)
finally:
    full_harness.close()

s1_summary = compute_summary_metrics(full_run["records"]["system1_pure"], warmup_discard=5)
s2_summary = compute_summary_metrics(full_run["records"]["system2_pure"], warmup_discard=5)
cascade_summary = compute_summary_metrics(
    full_run["records"]["cascade"],
    warmup_discard=5,
    system1_accuracy=s1_summary["accuracy"],
    system2_accuracy=s2_summary["accuracy"],
)
kpi_report = evaluate_kpis(
    cascade_summary,
    s2_summary,
    {"p95_reduction": 0.50, "token_savings": 0.60, "apgr_target": 0.95},
)

print("== G2: benchmark completo (200 amostras, α=0.85) ==")
for name, s in (
    ("system1_pure", s1_summary),
    ("system2_pure", s2_summary),
    ("cascade", cascade_summary),
):
    p = s["latency_percentiles_ms"]
    apgr_text = f"{s['apgr']:.3f}" if s["apgr"] is not None else "n/a"
    print(
        f"  {name}: n={s['n_requests']} p50={p['p50']:.1f}ms p95={p['p95']:.1f}ms"
        f" p99={p['p99']:.1f}ms | acc={s['accuracy']:.3f} f1={s['f1_macro']:.3f}"
        f" | taxaS2={s['system2_call_rate']:.2f} | decode={s['tokens_decode_total']}"
        f" | APGR={apgr_text}"
    )
for name, (passed, value), target in (
    ("KPI SC-002 redução de p95", kpi_report["p95_reduction"], ">=0.50"),
    ("KPI SC-004 economia de decode tokens", kpi_report["token_savings"], ">=0.60"),
    ("KPI SC-003 APGR", kpi_report["apgr"], ">=0.95"),
):
    print(f"  {name}: {value:.3f} vs alvo {target} → {'PASS' if passed else 'FAIL'}")
gate_f = s2_summary["json_success_rate"]
gate_verdict = "PASS" if gate_f >= JSON_GATE_TARGET else "FAIL"
print(
    f"  Gate F (re-medição oficial, 200 amostras): {gate_f:.3f}"
    f" vs alvo >= {JSON_GATE_TARGET:.2f} → {gate_verdict}"
)
s1_p95 = s1_summary["latency_percentiles_ms"]["p95"]
s1_verdict = "OK" if s1_p95 <= P95_SYSTEM1_TARGET_MS else "ACIMA"
print(
    f"  SC-001 (informativo): system1 p95 {s1_p95:.1f} ms"
    f" vs alvo <= {P95_SYSTEM1_TARGET_MS:.0f} ms → {s1_verdict}"
)

In [ ]:
# Story 3: sweep de α por replay sobre as medições do benchmark completo.
APGR_TARGET = 0.95  # kpis.apgr_target do config

sweep = sweep_alpha(full_run)
sweep_summaries = {
    alpha: compute_summary_metrics(
        records,
        warmup_discard=5,
        system1_accuracy=s1_summary["accuracy"],
        system2_accuracy=s2_summary["accuracy"],
    )
    for alpha, records in sweep.items()
}

print("== G3: sweep de α (replay) ==")
rates = []
for alpha in sorted(sweep_summaries):
    s = sweep_summaries[alpha]
    rates.append(s["system2_call_rate"])
    apgr_text = f"{s['apgr']:.3f}" if s["apgr"] is not None else "n/a"
    print(
        f"  α={alpha}: taxaS2={s['system2_call_rate']:.2f}"
        f" p95={s['latency_percentiles_ms']['p95']:.1f}ms"
        f" acc={s['accuracy']:.3f} decode={s['tokens_decode_total']} APGR={apgr_text}"
    )
monotonic = "OK" if rates == sorted(rates) else "VIOLADA"
print(f"  monotonicidade taxaS2: {monotonic} ({[round(r, 2) for r in rates]})")
qualifying = [
    a for a in sorted(sweep_summaries) if (sweep_summaries[a]["apgr"] or 0.0) >= APGR_TARGET
]
if qualifying:
    optimal = min(qualifying, key=lambda a: sweep_summaries[a]["system2_call_rate"])
else:
    optimal = max(sweep_summaries, key=lambda a: sweep_summaries[a]["apgr"] or 0.0)
print(f"  α ótimo: {optimal}")

In [ ]:
# Artefatos de Marco 1 (NFR-002 + Story 3): log bruto + curva de trade-off.
from datetime import UTC, datetime
from pathlib import Path

save_raw_results(
    Path("results/raw_benchmark_data.json"),
    meta={
        "model": system1.DEFAULT_MODEL,
        "base_url": system1.DEFAULT_BASE_URL,
        "frozen_samples": len(full_samples),
        "seed": 42,
        "alpha_default": 0.85,
        "alphas": sorted(sweep_summaries),
        "warmup_discard": 5,
        "generated_at_utc": datetime.now(UTC).isoformat(),
    },
    scenarios=full_run["records"],
    alpha_sweep={str(alpha): records for alpha, records in sweep.items()},
    summaries={
        "system1_pure": s1_summary,
        "system2_pure": s2_summary,
        "cascade": cascade_summary,
        "alpha_sweep": {str(alpha): s for alpha, s in sweep_summaries.items()},
    },
    kpis=kpi_report,
)
plot_tradeoff_curve(Path("results/latency_cost_tradeoff.png"), full_run, sweep_summaries)

print("== G4: artefatos ==")
for name in ("raw_benchmark_data.json", "latency_cost_tradeoff.png"):
    size = Path("results", name).stat().st_size
    print(f"  results/{name}: {size:,} bytes")

## Próxima fase

